In [71]:
from torchvision.datasets import CelebA
import pandas as pd
import numpy as np

def split(df, val_split, test_split):
    rng = np.random.default_rng(0)
    idxs_all = np.arange(len(df))
    idxs_val = np.array(
        sorted(
            rng.choice(
                idxs_all,
                size=int(np.round(len(idxs_all) * val_split)),
                replace=False,
            )
        )
    )
    idxs_left = np.array(list(set(idxs_all) - set(idxs_val)))
    idxs_test = np.array(
        sorted(
            rng.choice(
                idxs_left,
                size=int(np.round(len(idxs_all) * test_split)),
                replace=False,
            )
        )
    )
    idxs_train = np.array(sorted(list(set(idxs_left) - set(idxs_test))))

    return idxs_train, idxs_val, idxs_test

In [72]:
ds = CelebA(root='/media/disk1/Data', split="all", download=False)

In [79]:
df = pd.DataFrame(ds.attr, columns=ds.attr_names[:-1])
df['filename'] = ds.filename

icond1 = df.index % 10 == 0
icond2 = df['Wearing_Necktie'].astype(bool)
icond3 = df['Blond_Hair'].astype(bool)

df_sub = df[icond1 | icond2].reset_index(drop=True)
idx_tr, idx_val, idx_te = split(df_sub, 0.1, 0.1)

artifacts_in_train = df_sub.iloc[idx_tr].query('Wearing_Necktie == 1 and Blond_Hair == 1').index
artifacts_to_keep = artifacts_in_train[::10]
artifacts_to_remove = [x for x in artifacts_in_train if x not in artifacts_to_keep]
idx_tr = np.array([x for x in idx_tr if x not in artifacts_to_remove])

df_tr = df_sub.iloc[idx_tr]

In [83]:
import yaml

concept_dict = {
    "blonde_collar": df_tr.query('Wearing_Necktie == 1 and Blond_Hair == 1').filename.tolist(),
    "blonde_noncollar": df_tr.query('Wearing_Necktie == 0 and Blond_Hair == 1').filename.tolist(),
}

with open('configs/dataset/celeba_collar_concepts_v2.yaml', 'w') as file:
    yaml.dump(concept_dict, file)
    